# PokéVision Pi — entrenamiento en Google Colab

Entrena los **cuatro modelos** del proyecto con la GPU gratuita de Colab y descarga un
`models.zip` listo para copiar a la Raspberry Pi 4.

| Modelo | Qué aprende |
|---|---|
| `embedder` | qué carta es (aprendizaje contrastivo) |
| `condition` | auténtica / falsa / dañada |
| `grade` | subgrados tipo **PSA**: centrado, esquinas, bordes, superficie |
| `price` | valor futuro a 30/90/180 días |

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
Con GPU el entrenamiento completo tarda unos **12-18 minutos**; en CPU, más de una hora.


## 1 · Comprobar la GPU

In [ ]:
!nvidia-smi || echo "SIN GPU: ve a Entorno de ejecucion > Cambiar tipo de entorno > GPU"
import torch
print("torch", torch.__version__, "| cuda disponible:", torch.cuda.is_available())

## 2 · Clonar el repositorio e instalar dependencias

In [ ]:
!git clone -q https://github.com/caesar-dat-com/pokevision-pi.git
%cd pokevision-pi
!pip -q install onnx onnxruntime joblib
print("listo")

## 3 · Descargar el dataset desde TCGdex

Unas 4.500 cartas de los 30 sets más recientes: imágenes oficiales + precios reales de
Cardmarket. Tarda ~4 minutos. Sube `--sets` si quieres un catálogo más grande
(60 sets ≈ 9.000 cartas, ~8 min).

In [ ]:
!python scripts/fetch_dataset.py --sets 30 --workers 32
!ls data/images | wc -l
!wc -l data/cards.jsonl

## 4 · Identificador de cartas (aprendizaje contrastivo)

No hay fotos reales etiquetadas de cada carta: solo el escaneo oficial. Así que de cada
carta se generan dos "fotos" sintéticas distintas (perspectiva, brillo, reflejos, ruido)
y la red aprende a reconocerlas como la misma carta. Con GPU se puede subir el batch,
que es justo lo que mejora el contrastivo.

In [ ]:
!python vision/train/train_embedder.py --epochs 12 --batch 192 --workers 2

## 5 · Autenticidad y estado (auténtica / falsa / dañada)

In [ ]:
!python vision/train/train_condition.py --epochs 8 --batch 192 --per-class 2 --workers 2

## 6 · Calificación tipo PSA

Se aplica desgaste sintético de **severidad conocida** en los cuatro ejes que puntúa PSA
y la red aprende a leer esa severidad desde la imagen:

    etiqueta_subgrado = 10 - 9 * severidad

El split es **por carta**: ninguna carta del test se vio entrenando.

In [ ]:
!python vision/train/train_grade.py --epochs 8 --batch 192 --per-card 3 --workers 2

## 7 · Modelo de valor futuro (CPU, segundos)

In [ ]:
!python pricing/train_price.py

## 8 · Exportar a ONNX y construir el índice

La Raspberry no lleva PyTorch: solo `onnxruntime`. Aquí se convierten los pesos y se
precalculan los vectores de todas las cartas del catálogo.

In [ ]:
!python vision/build_index.py
!ls -lh vision/models/

## 9 · Ver las métricas obtenidas

In [ ]:
import json, pathlib
for f in ["vision/models/embedder_metrics.txt",
          "vision/models/condition_metrics.json",
          "vision/models/grade_metrics.json",
          "pricing/models/price_metrics.json"]:
    p = pathlib.Path(f)
    if p.exists():
        print("="*60); print(f)
        print(p.read_text()[:1200])

## 10 · Descargar `models.zip` para la Raspberry Pi

Descomprime este zip en la raíz del repo clonado en la Pi y arranca el servicio.
`data/images/` **no** hace falta allí: el índice ya lleva los vectores.

In [ ]:
!zip -qr models.zip vision/models/embedder.onnx vision/models/condition.onnx \
      vision/models/grade.onnx vision/models/index.npy vision/models/index_meta.json \
      vision/models/*metrics* pricing/models data/cards.jsonl
!du -h models.zip
from google.colab import files
files.download('models.zip')

---
### En la Raspberry Pi 4

```bash
git clone https://github.com/caesar-dat-com/pokevision-pi.git
cd pokevision-pi
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements-pi.txt
unzip ~/models.zip            # el zip descargado de Colab
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

Detalles de despliegue, kiosco y rendimiento: `docs/RASPBERRY.md`.
